<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB_WEEK4_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주제** 자동차 보험사기 탐지 해커톤

https://dacon.io/competitions/official/236441/codeshare/12664?page=1&dtype=recent

**코드 흐름**

**EDA**
- 불균형 데이터 여부 파악
- 수치형 변수 상관 분석
   - 강한 상관 관계가 있는 변수를 파악. 다른 수치형 변수와의 range 차이가 심할 경우 일부 칼럼만을 학습에 이용.
- 범주형 변수 상관 분석
   - 각 범주와 종속변수와의 관계 파악.
   - **카이제곱 검정**을 이용하여 각 범주형 변수와 타켓 변수와의 독립성 여부 파악. 유의한 관계가 없을 경우 분석에서 제외.

**전처리**
- 결측치가 전체 데이터 셋에 비해 적을 경우 결측치 행 삭제
- 두 값뿐인 칼럼은 0과 1로 매핑.
- 수치형 칼럼에 StandardScaling 적용
- 범주형 칼럼에 one hot encoding 적용
- 요일 변수에 **Cyclic encoding** 적용.
- SMOTE의 Oversampling 기법을 적용해 불균형 데이터 처리.

**모델링**
- RandomForest, XGBoost, LightGBM 학습
- GridSearch로 최적 하이퍼 파라미터 적용
- LogisticRegression 학습 결과 F1 점수 향상.

**새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

수치형 변수와 타깃값의 상관성을 알아볼 때 상관 히트맵을 만들 듯이, 범주형 변수와 타깃값의 독립성을 알아보기 위해 카이제곱 검정을 이용할 수 있다는 점을 알았다. 이를 프로젝트에 적용해 보아도 좋을 것 같다고 생각했다. 또, 요일 변수에 cyclic encoding을 적용했는데, 이를 과제와 세션에서 보지 못한 코드라 검색해 본 결과 순환하는 숫자를 인코딩하는 방법이라는 점을 새롭게 알게 되었다.


In [ ]:
# 카이제곱 검정을 이용해 범주형 변수의 독립성 여부 판단

from scipy.stats import chi2_contingency

for col in cat_cols:
    contingency = pd.crosstab(train[col], train['fraud'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"{col}: p-value = {p:.4f}")

In [ ]:
# 요일과 월 Cyclic encoding

day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}
train['day_num'] = train['claim_day_of_week'].map(day_map)
test['day_num'] = test['claim_day_of_week'].map(day_map)

train['day_sin'] = np.sin(2 * np.pi * train['day_num'] / 7)
train['day_cos'] = np.cos(2 * np.pi * train['day_num'] / 7)

test['day_sin'] = np.sin(2 * np.pi * test['day_num'] / 7)
test['day_cos'] = np.cos(2 * np.pi * test['day_num'] / 7)

month_map = {
    'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
    'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
}
train['month_num'] = train['month'].map(month_map)
test['month_num'] = test['month'].map(month_map)

train['month_sin'] = np.sin(2 * np.pi * train['month_num'] / 12)
train['month_cos'] = np.cos(2 * np.pi * train['month_num'] / 12)

test['month_sin'] = np.sin(2 * np.pi * test['month_num'] / 12)
test['month_cos'] = np.cos(2 * np.pi * test['month_num'] / 12)

train = train.drop(['claim_day_of_week', 'month', 'day_num', 'month_num'], axis=1)
test = test.drop(['claim_day_of_week', 'month', 'day_num', 'month_num'], axis=1)

In [ ]:
# SMOTE를 이용한 OverSampling
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = train.drop('fraud', axis=1)
y = train['fraud']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)